# COE Scenario Generation
Generate creative scenarios for error chains using GPT-4o batch API.


In [1]:
import os
import sys
import json
from pathlib import Path
from tokens import openai_key

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm.metrics.utils.e_gen_scenario import COEScenarioGenerator


In [2]:
# Initialize generator
MODEL_NAME = "Qwen3-VL-8B-Instruct"
DATASET = "fvqa"  # or "aokvqa"

gen = COEScenarioGenerator(
    dataset_name=DATASET,
    model_name=MODEL_NAME,
    openai_key=openai_key
)


COEScenarioGenerator: Qwen3-VL-8B-Instruct/fvqa, 20 batches


## Step 1: Load COE results and prepare batch input


In [3]:
# Load COE prediction results
coe_path = f"results/pred_postedit/baseline/{MODEL_NAME}/{DATASET}/coe_prediction.json"
with open(coe_path, "r") as f:
    coe_results = json.load(f)

print(f"Loaded {len(coe_results)} COE results from {coe_path}")

# Check how many have error chains
with_errors = [r for r in coe_results if gen._get_error_chains(r.get("coe_pred", {}))]
print(f"Samples with error chains: {len(with_errors)}/{len(coe_results)}")

# Preview a few
for r in with_errors[:3]:
    chains = gen._get_error_chains(r.get("coe_pred", {}))
    print(f"uid={r['uid']}: {len(chains)} error chains")

# Preview the prompt for one sample
sample = with_errors[0]
chains = gen._get_error_chains(sample.get("coe_pred", {}))
print(f"Sample has {len(chains)} error chains. Showing first:\n")
print("=== PROMPT ===")
print(gen.format_prompt(chains[0]["chain"]))

Loaded 857 COE results from results/pred_postedit/baseline/Qwen3-VL-8B-Instruct/fvqa/coe_prediction.json
Samples with error chains: 371/857
uid=11: 2 error chains
uid=18: 4 error chains
uid=19: 4 error chains
Sample has 2 error chains. Showing first:

=== PROMPT ===
Given these visual facts:
"The pizzas have a topping that is light in color and melted."

Generate 3 different creative scenarios where ALL these facts would be visually true.

Requirements:
- Each scenario should be a distinct visual setting, in 2-3 sentences
- Be creative but plausible
- Describe what would be visible in the image

Examples:
Visual facts: "A person is standing on a board. There are waves around."
1. A surfer rides a wave at a tropical beach during sunset. The ocean is blue with white foam. Palm trees line the shore in the background.
2. A wakeboarder is pulled behind a speedboat on a lake. The boat creates a large wake. Mountains are visible in the distance.
3. A paddleboarder balances on calm ocean water

In [4]:
# Generate batch request files (uncomment to run)
gen.run_input(coe_results, max_sentences=None)  # or max_sentences=5 to filter


Created 1709 requests in data/coe_gen/Qwen3-VL-8B-Instruct/fvqa/requests/batch.jsonl


## Step 2: Test with ONE batch before firing all


In [5]:
# Check batch files created
batch_files = list(gen.batch_dir.glob("batch_*.jsonl"))
print(f"Batch files: {len(batch_files)}")
for bf in sorted(batch_files)[:5]:
    with open(bf) as f:
        n_lines = len(f.readlines())
    print(f"  {bf.name}: {n_lines} requests")
    
# Preview first request in batch_0
batch_0 = gen.batch_dir / "batch_0.jsonl"
if batch_0.exists():
    with open(batch_0) as f:
        first_req = json.loads(f.readline())
    print(json.dumps(first_req, indent=2)[:1000])  # truncate if too long



Batch files: 20
  batch_0.jsonl: 86 requests
  batch_1.jsonl: 86 requests
  batch_10.jsonl: 86 requests
  batch_11.jsonl: 86 requests
  batch_12.jsonl: 86 requests
{
  "custom_id": "11_[1]",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4o-mini",
    "messages": [
      {
        "role": "system",
        "content": "You generate creative visual scenarios from given facts."
      },
      {
        "role": "user",
        "content": "Given these visual facts:\n\"The pizzas have a topping that is light in color and melted.\"\n\nGenerate 3 different creative scenarios where ALL these facts would be visually true.\n\nRequirements:\n- Each scenario should be a distinct visual setting, in 2-3 sentences\n- Be creative but plausible\n- Describe what would be visible in the image\n\nExamples:\nVisual facts: \"A person is standing on a board. There are waves around.\"\n1. A surfer rides a wave at a tropical beach during sunset. The ocean is blue with white f

In [8]:
# Submit ONLY batch 0 first (uncomment to run)
gen._run_request_batch(0)


Batch 0: batch_6945d11f501c8190a64b6d83adf012e6


In [9]:
# Check batch 0 status
meta_0 = gen.meta_dir / "meta_0.json"
if meta_0.exists():
    with open(meta_0) as f:
        meta = json.load(f)
    job = gen.client.batches.retrieve(meta["job_id"])
    print(f"Batch 0 status: {job.status}")
    print(f"  request_counts: {job.request_counts}")
else:
    print("Batch 0 not submitted yet")


Batch 0 status: in_progress
  request_counts: BatchRequestCounts(completed=0, failed=0, total=86)


In [ ]:
# Get batch 0 results (after it completes)
try:
    results_0 = gen._get_response_batch(0)
    print(f"Got {len(results_0)} results from batch 0")
    
    # Preview first few
    for r in results_0[:3]:
        print(f"\nuid={r['uid']}:")
        for i, s in enumerate(r['scenarios'], 1):
            print(f"  {i}. {s}")
except Exception as e:
    print(f"Error: {e}")


## Step 3: Fire all batches (after testing batch 0)


In [ ]:
# Submit all remaining batches (uncomment to run)
# gen.run_request()


In [ ]:
# Check status of all submitted batches
for b in range(gen.n_batches):
    meta_path = gen.meta_dir / f"meta_{b}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        try:
            job = gen.client.batches.retrieve(meta["job_id"])
            print(f"Batch {b}: {job.status}")
        except Exception as e:
            print(f"Batch {b}: error - {e}")


In [ ]:
# Resubmit failed batches if needed (uncomment and modify list)
# gen.resubmit_request([0, 1, 2])  # list of batch indices to resubmit


## Step 4: Get all results


In [ ]:
# Get all scenarios (after all batches complete)
all_scenarios = gen.get_scenarios()
print(f"Total scenarios: {len(all_scenarios)}")

# Preview results
for r in all_scenarios[:3]:
    print(f"\nuid={r['uid']}, indices={r['indices']}:")
    for i, s in enumerate(r['scenarios'], 1):
        print(f"  {i}. {s}")
